Código 305: Preparación de Features y Pipeline de Modelado

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ===========================================================================
# 305.1 RECONSTRUCCIÓN Y SANEAMIENTO DE TIPOS (DATA TYPING)
# ===========================================================================
path_raw = r'C:\Tesis_ML\raw_data\nyswcb_claims.csv'
df = pd.read_csv(path_raw, low_memory=False)

# Limpieza de cabeceras [cite: 9]
df.columns = [c.strip().replace('\r', '').replace('\n', '') for c in df.columns]

# Saneamiento de la variable salarial (AWW) 
# Eliminamos símbolos de moneda y comas para permitir la conversión a float
df['Average Weekly Wage (AWW)'] = (
    df['Average Weekly Wage (AWW)']
    .replace('[\$,]', '', regex=True)
    .astype(float)
    .fillna(0)
)

# Normalización Temporal [cite: 40]
df['Accident Date'] = pd.to_datetime(df['Accident Date'], errors='coerce')
df['Year'] = df['Accident Date'].dt.year
df['Month'] = df['Accident Date'].dt.month

# Filtrado de ventana de madurez (2010-2024)
df_ts = df[(df['Year'] >= 2010) & (df['Year'] <= 2024)].copy()

# ===========================================================================
# 305.2 TRANSFORMACIÓN Y DEFINICIÓN DE CLASE (TARGET)
# ===========================================================================
# 1. Estabilización de Varianza (Log-Transform) [cite: 135]
df_ts['AWW_Log'] = np.log1p(df_ts['Average Weekly Wage (AWW)'])

# 2. Señal de Silencio: Nulos médicos [cite: 62, 96]
df_ts['Missing_Medical_Data'] = df_ts['OIICS Nature of Injury Code'].isnull().astype(int)

# 3. Target: Éxito del establecimiento (Has_ANCR) 
df_ts['Has_ANCR'] = df_ts['Interval Assembled to ANCR'].notnull().astype(int)

# ===========================================================================
# 305.3 PREPARACIÓN DEL BENCHMARK
# ===========================================================================
features = ['AWW_Log', 'Carrier Type', 'District Name', 'Year', 'Month', 'Missing_Medical_Data']
target = 'Has_ANCR'

# Eliminación de nulos residuales en variables de contexto
df_ml = df_ts.dropna(subset=['Carrier Type', 'District Name'])

X = df_ml[features]
y = df_ml[target]

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"--- DATASET SANEADO Y LISTO ---")
print(f"Clase a predecir (Has_ANCR): {y.mean()*100:.2f}% de casos establecidos.")

<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\$'
C:\Users\julia\AppData\Local\Temp\ipykernel_166212\1736881418.py:18: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)


--- DATASET SANEADO Y LISTO ---
Clase a predecir (Has_ANCR): 42.62% de casos establecidos.
